**INITIAL PARAMETERS**

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import pickle
import os
from tqdm import tqdm
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa
from tqdm.notebook import trange
import pickle

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% DEFINE PARAMETERS HERE %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
base_path = "F:"                          # Base path where trace datasets are stored
num_iterations = 11                       # Number of iterations per capture to build one full trace
                                          # (3 for unmasked, 11 for masked implementations)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

complete_traces_list = []
trace_length = 24400 * num_iterations     # Length of a full concatenated trace
traces_per_folder = 10
num_traces_per_folder = 10


**TRACE LOADING + UNIVARIATE TVLA (1ST, 2ND, 3RD ORDER)**

In [ ]:
import numpy as np
from tqdm import tqdm
import os
import h5py

n_chunks = num_iterations                     # Number of sub-traces concatenated per full trace
trace_length = 24400 * n_chunks               # Full trace length
FOLDERS_LIST = ["DATASET_KECCAK_TIDOM_2"]

MAX_FILES = 200                              # Maximum number of HDF5 files to process

# -----------------------------
# Step 1: Compute global mean per sample point
# -----------------------------
sum_all = np.zeros(trace_length, dtype=np.float64)
count_all = 0

for folder_name in FOLDERS_LIST:
    folder_path = os.path.join(base_path, folder_name)

    h5_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".h5")])[:MAX_FILES]

    for h5_file in tqdm(h5_files, desc="Pass 1: global mean"):
        h5_path = os.path.join(folder_path, h5_file)

        try:
            with h5py.File(h5_path, "r") as f:
                traces = f["traces"][:]      # Shape: (num_traces, trace_length)
        except Exception as e:
            print(f"Error opening {h5_path}: {e}")
            continue

        sum_all += np.sum(traces, axis=0)
        count_all += traces.shape[0]

mean_global = sum_all / count_all
print("Global mean per trace sample computed")

# -----------------------------
# Step 2: Accumulate TVLA statistics (1st, 2nd and 3rd order)
# -----------------------------
sum_x_f = np.zeros(trace_length, dtype=np.float64)
sum_x_r = np.zeros(trace_length, dtype=np.float64)
sum_x2_f = np.zeros(trace_length, dtype=np.float64)
sum_x2_r = np.zeros(trace_length, dtype=np.float64)
sum_x3_f = np.zeros(trace_length, dtype=np.float64)
sum_x3_r = np.zeros(trace_length, dtype=np.float64)

sum_y_f = np.zeros(trace_length, dtype=np.float64)
sum_y_r = np.zeros(trace_length, dtype=np.float64)
sum_y2_f = np.zeros(trace_length, dtype=np.float64)
sum_y2_r = np.zeros(trace_length, dtype=np.float64)

count_fixed = 0
count_random = 0

for folder_name in FOLDERS_LIST:
    folder_path = os.path.join(base_path, folder_name)
    h5_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".h5")])[:MAX_FILES]

    for h5_file in tqdm(h5_files, desc="Pass 2: TVLA statistics"):
        h5_path = os.path.join(folder_path, h5_file)

        try:
            with h5py.File(h5_path, "r") as f:
                traces = f["traces"][:]
        except Exception as e:
            print(f"Error opening {h5_path}: {e}")
            continue

        for i, trace in enumerate(traces):
            x = trace
            y = (trace - mean_global) ** 2
            z = (trace - mean_global) ** 3

            if i % 2 == 0:     # Fixed input set
                sum_x_f += x
                sum_x2_f += x * x
                sum_x3_f += x * x * x
                sum_y_f += y
                sum_y2_f += y * y
                count_fixed += 1
            else:              # Random input set
                sum_x_r += x
                sum_x2_r += x * x
                sum_x3_r += x * x * x
                sum_y_r += y
                sum_y2_r += y * y
                count_random += 1

# -----------------------------
# First-order TVLA
# -----------------------------
mean_x_f = sum_x_f / count_fixed
mean_x_r = sum_x_r / count_random
var_x_f = (sum_x2_f / count_fixed) - mean_x_f**2
var_x_r = (sum_x2_r / count_random) - mean_x_r**2

t_num_1 = mean_x_f - mean_x_r
t_den_1 = np.sqrt(var_x_f / count_fixed + var_x_r / count_random)
t_values_1 = np.nan_to_num(t_num_1 / t_den_1)

print("First-order TVLA completed")
print("Max |t| (1st order):", np.max(np.abs(t_values_1)))

# -----------------------------
# Second-order TVLA
# -----------------------------
mean_y_f = sum_y_f / count_fixed
mean_y_r = sum_y_r / count_random
var_y_f = (sum_y2_f / count_fixed) - mean_y_f**2
var_y_r = (sum_y2_r / count_random) - mean_y_r**2

t_num_2 = mean_y_f - mean_y_r
t_den_2 = np.sqrt(var_y_f / count_fixed + var_y_r / count_random)
t_values_2 = np.nan_to_num(t_num_2 / t_den_2)

print("Second-order TVLA completed")
print("Max |t| (2nd order):", np.max(np.abs(t_values_2)))

# -----------------------------
# Third-order TVLA
# -----------------------------
mean_g1 = sum_x_r / count_random
mean_g2 = sum_x_f / count_fixed
var_g1 = (sum_x2_r / count_random) - (mean_g1 ** 2)
var_g2 = (sum_x2_f / count_fixed) - (mean_g2 ** 2)
m3_g1 = (sum_x3_r / count_random) - 3 * mean_g1 * var_g1 - (mean_g1 ** 3)
m3_g2 = (sum_x3_f / count_fixed) - 3 * mean_g2 * var_g2 - (mean_g2 ** 3)

t_num_3 = m3_g1 - m3_g2
t_den_3 = np.sqrt((var_g1**3) / count_random + (var_g2**3) / count_fixed)
t_values_3 = t_num_3 / t_den_3

print("Third-order TVLA completed")
print("Max |t| (3rd order):", np.max(np.abs(t_values_3)))


**TVLA VISUALIZATION**

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t_values_1, label='TVLA 1st order')
plt.title('Keccak TVLA - 1st order',fontsize="16")
plt.axhline(y=4.5, color='green', linestyle='--', label='4.5 leakage threshold')
plt.axhline(y=-4.5, color='green', linestyle='--')
plt.axhline(y=6.6, color='red', linestyle='--', label='6.6 leakage threshold')
plt.axhline(y=-6.6, color='red', linestyle='--')
plt.xlabel('Samples',fontsize="14")
plt.ylabel('t-values',fontsize="14")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.legend(loc='lower right')
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t_values_2, label='TVLA 2nd order')
plt.title('Keccak TVLA - 2nd order',fontsize="16")
plt.axhline(y=4.5, color='green', linestyle='--', label='4.5 leakage threshold')
plt.axhline(y=-4.5, color='green', linestyle='--')
plt.axhline(y=6.6, color='red', linestyle='--', label='6.6 leakage threshold')
plt.axhline(y=-6.6, color='red', linestyle='--')
plt.xlabel('Samples',fontsize="14")
plt.ylabel('t-values',fontsize="14")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t_values_3, label='TVLA 3rd order')
plt.title('Keccak TVLA - 3rd order',fontsize="16")
plt.axhline(y=4.5, color='green', linestyle='--', label='4.5 leakage threshold')
plt.axhline(y=-4.5, color='green', linestyle='--')
plt.axhline(y=6.6, color='red', linestyle='--', label='6.6 leakage threshold')
plt.axhline(y=-6.6, color='red', linestyle='--')
plt.xlabel('Samples',fontsize="14")
plt.ylabel('t-values',fontsize="14")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.legend(loc='lower right')
plt.show()